# AOU Data Extraction and Split Builder

Independent data-prep workflow for All of Us Workbench.

This notebook: 
- runs direct BigQuery extraction queries
- previews pandas DataFrames
- builds participant-level train/val/test splits
- saves Parquet artifacts (optional NPZ + bucket upload)

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

print(f"numpy={np.__version__}")
print(f"pandas={pd.__version__}")

In [ ]:
CFG: dict[str, Any] = {
    "seed": 42,
    "max_persons": 5000,
    "preview_rows": 5,
    "split_ratios": {"train": 0.7, "val": 0.15, "test": 0.15},
    "min_measurement_events_per_person": 1,
    "measurement_concept_ids": [],
    "label_condition_concept_ids": [],
    "output_local_dir": "/home/jupyter/data/preprocessed",
    "output_bucket_prefix": "",
    "write_npz": False,
    "dry_run": False,
}

workspace_bucket = os.getenv("WORKSPACE_BUCKET", "").strip()
workspace_cdr = os.getenv("WORKSPACE_CDR", "").strip()

CFG["workspace_bucket"] = workspace_bucket
CFG["workspace_cdr"] = workspace_cdr
if not str(CFG.get("output_bucket_prefix", "")).strip() and workspace_bucket:
    CFG["output_bucket_prefix"] = f"{workspace_bucket.rstrip('/')}/data/preprocessed_parquet"

if not workspace_cdr:
    raise EnvironmentError("WORKSPACE_CDR is empty. Start this notebook in AOU Workbench.")

print("Detected WORKSPACE_BUCKET:", workspace_bucket)
print("Detected WORKSPACE_CDR:", workspace_cdr)
print(json.dumps(CFG, indent=2))

In [ ]:
def run_bq_query(name: str, query: str, dry_run: bool = False) -> pd.DataFrame:
    print(f"[{name}] query length: {len(query)} characters")
    if dry_run:
        print(f"[{name}] dry_run=True, skipping execution")
        return pd.DataFrame()
    return pd.read_gbq(
        query,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )


def validated_split_counts(n: int, ratios: dict[str, float]) -> tuple[int, int, int]:
    train_r = float(ratios.get("train", 0.7))
    val_r = float(ratios.get("val", 0.15))
    test_r = float(ratios.get("test", 0.15))
    if min(train_r, val_r, test_r) < 0:
        raise ValueError("split ratios must be non-negative")
    total = train_r + val_r + test_r
    if total <= 0:
        raise ValueError("split ratios must sum to > 0")
    train_r, val_r, test_r = train_r / total, val_r / total, test_r / total
    n_train = int(round(n * train_r))
    n_val = int(round(n * val_r))
    n_test = n - n_train - n_val
    while n_train + n_val + n_test > n:
        if n_test > 0:
            n_test -= 1
        elif n_val > 0:
            n_val -= 1
        else:
            n_train -= 1
    while n_train + n_val + n_test < n:
        n_train += 1
    return n_train, n_val, n_test


def save_frame(df: pd.DataFrame, path: Path, allow_csv_fallback: bool = True) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        return path
    except Exception as exc:
        if not allow_csv_fallback:
            raise
        csv_path = path.with_suffix(".csv")
        print(f"Parquet write failed for {path.name}: {exc}. Falling back to {csv_path.name}")
        df.to_csv(csv_path, index=False)
        return csv_path

In [ ]:
workspace_cdr = CFG["workspace_cdr"]
cohort_limit = int(CFG["max_persons"])

measurement_filter_sql = ""
measurement_concept_ids = list(CFG.get("measurement_concept_ids", []))
if measurement_concept_ids:
    ids_sql = ", ".join(str(int(x)) for x in measurement_concept_ids)
    measurement_filter_sql = f"AND m.measurement_concept_id IN ({ids_sql})"

dataset_aou_cohort_sql = f"""
SELECT
  person_id
FROM `{workspace_cdr}.cb_search_person`
WHERE has_ehr_data = 1
  AND has_physical_measurement_data = 1
LIMIT {cohort_limit}
"""

dataset_aou_person_sql = f"""
WITH cohort AS (
  {dataset_aou_cohort_sql}
)
SELECT
  p.person_id,
  p.birth_datetime,
  p.gender_concept_id,
  p.race_concept_id,
  p.ethnicity_concept_id
FROM `{workspace_cdr}.person` p
JOIN cohort c
  ON p.person_id = c.person_id
"""

dataset_aou_measurement_sql = f"""
WITH cohort AS (
  {dataset_aou_cohort_sql}
)
SELECT
  m.person_id,
  m.measurement_datetime,
  m.measurement_concept_id,
  m.value_as_number
FROM `{workspace_cdr}.measurement` m
JOIN cohort c
  ON m.person_id = c.person_id
WHERE m.value_as_number IS NOT NULL
  {measurement_filter_sql}
"""

dataset_aou_condition_sql = f"""
WITH cohort AS (
  {dataset_aou_cohort_sql}
)
SELECT
  co.person_id,
  co.condition_start_datetime,
  co.condition_concept_id
FROM `{workspace_cdr}.condition_occurrence` co
JOIN cohort c
  ON co.person_id = c.person_id
"""

print("Prepared SQL queries")
print("- cohort", len(dataset_aou_cohort_sql))
print("- person", len(dataset_aou_person_sql))
print("- measurement", len(dataset_aou_measurement_sql))
print("- condition", len(dataset_aou_condition_sql))

In [ ]:
cohort_df = run_bq_query("cohort", dataset_aou_cohort_sql, dry_run=bool(CFG["dry_run"]))
person_df = run_bq_query("person", dataset_aou_person_sql, dry_run=bool(CFG["dry_run"]))
measurement_df = run_bq_query("measurement", dataset_aou_measurement_sql, dry_run=bool(CFG["dry_run"]))
condition_df = run_bq_query("condition", dataset_aou_condition_sql, dry_run=bool(CFG["dry_run"]))

if bool(CFG["dry_run"]):
    print("dry_run=True, stop here after query validation")
else:
    print("DataFrame shapes:")
    print("cohort:", cohort_df.shape)
    print("person:", person_df.shape)
    print("measurement:", measurement_df.shape)
    print("condition:", condition_df.shape)
    n = int(CFG["preview_rows"])
    print("\nPreview: person")
    display(person_df.head(n))
    print("\nPreview: measurement")
    display(measurement_df.head(n))
    print("\nPreview: condition")
    display(condition_df.head(n))

In [ ]:
if bool(CFG["dry_run"]):
    raise RuntimeError("Set CFG['dry_run']=False to build files")
if person_df.empty:
    raise RuntimeError("person query returned zero rows")

if "birth_datetime" in person_df.columns:
    person_df["birth_datetime"] = pd.to_datetime(person_df["birth_datetime"], utc=True, errors="coerce")
if "measurement_datetime" in measurement_df.columns:
    measurement_df["measurement_datetime"] = pd.to_datetime(measurement_df["measurement_datetime"], utc=True, errors="coerce")
if "condition_start_datetime" in condition_df.columns:
    condition_df["condition_start_datetime"] = pd.to_datetime(condition_df["condition_start_datetime"], utc=True, errors="coerce")

label_ids_filter = list(CFG.get("label_condition_concept_ids", []))
target_conditions = condition_df[condition_df["condition_concept_id"].isin(label_ids_filter)] if label_ids_filter else condition_df
label_df = pd.DataFrame({"person_id": person_df["person_id"].drop_duplicates()})
label_ids = set(target_conditions["person_id"].dropna().astype(int).tolist())
label_df["label"] = label_df["person_id"].isin(label_ids).astype(np.int32)

m = measurement_df.dropna(subset=["person_id", "value_as_number"]).copy()
if m.empty:
    raise RuntimeError("measurement query returned no usable value_as_number rows")
m["value_as_number"] = pd.to_numeric(m["value_as_number"], errors="coerce")
m = m.dropna(subset=["value_as_number"])

measurement_features = m.groupby("person_id").agg(
    measurement_count=("value_as_number", "size"),
    measurement_mean=("value_as_number", "mean"),
    measurement_std=("value_as_number", "std"),
    measurement_min=("value_as_number", "min"),
    measurement_max=("value_as_number", "max"),
    last_measurement_value=("value_as_number", "last"),
    n_unique_measurement_concepts=("measurement_concept_id", "nunique"),
    first_measurement_time=("measurement_datetime", "min"),
    last_measurement_time=("measurement_datetime", "max"),
).reset_index()
measurement_features["measurement_span_days"] = (
    measurement_features["last_measurement_time"] - measurement_features["first_measurement_time"]
).dt.days.fillna(0).astype(np.float32)

person_features = person_df[["person_id", "birth_datetime", "gender_concept_id", "race_concept_id", "ethnicity_concept_id"]].drop_duplicates(subset=["person_id"]).copy()
today = pd.Timestamp.now(tz="UTC")
person_features["age_years"] = (
    (today - person_features["birth_datetime"]).dt.total_seconds() / (365.25 * 24 * 3600)
).clip(lower=0)

feature_df = person_features.merge(measurement_features, on="person_id", how="inner")
feature_df = feature_df.merge(label_df, on="person_id", how="left")
feature_df["label"] = feature_df["label"].fillna(0).astype(np.int32)
feature_df = feature_df[feature_df["measurement_count"] >= int(CFG.get("min_measurement_events_per_person", 1))].copy()

for col in ["gender_concept_id", "race_concept_id", "ethnicity_concept_id"]:
    feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce").fillna(0).astype(np.float32)

numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
feature_df[numeric_cols] = feature_df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

ids = feature_df["person_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(int(CFG["seed"]))
rng.shuffle(ids)

n_train, n_val, n_test = validated_split_counts(len(ids), dict(CFG["split_ratios"]))
train_ids = set(ids[:n_train])
val_ids = set(ids[n_train:n_train + n_val])

feature_df["split"] = "test"
feature_df.loc[feature_df["person_id"].isin(train_ids), "split"] = "train"
feature_df.loc[feature_df["person_id"].isin(val_ids), "split"] = "val"

train_df = feature_df[feature_df["split"] == "train"].copy()
val_df = feature_df[feature_df["split"] == "val"].copy()
test_df = feature_df[feature_df["split"] == "test"].copy()

output_dir = Path(str(CFG["output_local_dir"]))
output_dir.mkdir(parents=True, exist_ok=True)
saved_paths: list[Path] = []
saved_paths.append(save_frame(train_df, output_dir / "train.parquet"))
saved_paths.append(save_frame(val_df, output_dir / "val.parquet"))
saved_paths.append(save_frame(test_df, output_dir / "test.parquet"))
saved_paths.append(save_frame(feature_df, output_dir / "full_cohort.parquet"))

meta = {
    "row_counts": {
        "full_cohort": int(len(feature_df)),
        "train": int(len(train_df)),
        "val": int(len(val_df)),
        "test": int(len(test_df)),
    },
    "positive_rate": {
        "full_cohort": float(feature_df["label"].mean()),
        "train": float(train_df["label"].mean()) if len(train_df) else None,
        "val": float(val_df["label"].mean()) if len(val_df) else None,
        "test": float(test_df["label"].mean()) if len(test_df) else None,
    },
}
meta_path = output_dir / "dataset_metadata.json"
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
saved_paths.append(meta_path)

if bool(CFG.get("write_npz", False)):
    feature_cols = [
        c for c in feature_df.columns
        if c not in {"person_id", "split", "label", "birth_datetime", "first_measurement_time", "last_measurement_time"}
    ]
    for split_name, split_frame in {"train": train_df, "val": val_df, "test": test_df}.items():
        x = split_frame[feature_cols].to_numpy(dtype=np.float32)
        missing = np.isnan(x).astype(np.float32)
        x = np.nan_to_num(x, nan=0.0).astype(np.float32)
        payload = {
            "x_dense": x[:, None, :],
            "x_sparse": np.zeros((x.shape[0], 1, 0), dtype=np.float32),
            "missing_mask": missing[:, None, :],
            "y": split_frame["label"].to_numpy(dtype=np.float32),
        }
        out = output_dir / f"{split_name}.npz"
        np.savez_compressed(out, **payload)
        saved_paths.append(out)

bucket_prefix = str(CFG.get("output_bucket_prefix", "")).strip().rstrip("/")
if bucket_prefix:
    for local_path in saved_paths:
        target_uri = f"{bucket_prefix}/{local_path.name}"
        cmd = ["gsutil", "cp", str(local_path), target_uri]
        print("Running:", " ".join(cmd))
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode != 0:
            print(f"Warning: upload failed for {local_path.name}; exit={proc.returncode}")

print("Built feature table:", feature_df.shape)
print("Split sizes:", {"train": train_df.shape, "val": val_df.shape, "test": test_df.shape})
print("Saved local artifacts:")
for p in saved_paths:
    print("-", p)
display(feature_df.head(int(CFG["preview_rows"])))

## Next Step

Use Parquet splits first for feature iteration. Enable NPZ export only when your training pipeline needs tensor inputs.